# Aula 10 — Estatística básica e valores ausentes

**Módulo 3 — NumPy e Computação Numérica**

## Objetivos da aula

- Calcular estatísticas descritivas com NumPy: média, mediana, soma, mínimo, máximo, variância e desvio padrão.
- Entender o que é um valor `NaN` e por que ele aparece em dados reais.
- Identificar e tratar valores ausentes de forma inicial.

---

## 1. Estatísticas descritivas com NumPy

Dado um conjunto de leituras, é comum precisarmos resumir a informação em poucos números: valor central, dispersão, extremos. O NumPy oferece essas funções prontas.

In [7]:
import numpy as np

leituras_temperatura = np.array([72.0, 76.5, 91.0, 68.3, 85.2, 74.1, 79.8])

print("Soma:            ", leituras_temperatura.sum())
print("Média:           ", leituras_temperatura.mean())
print("Mediana:         ", np.median(leituras_temperatura))
print("Mínimo:          ", leituras_temperatura.min())
print("Máximo:          ", leituras_temperatura.max())
print("Desvio padrão:   ", round(leituras_temperatura.std(), 2))
print("Variância:       ", round(leituras_temperatura.var(), 2))


Soma:             546.9
Média:            78.12857142857142
Mediana:          76.5
Mínimo:           68.3
Máximo:           91.0
Desvio padrão:    7.29
Variância:        53.07


**O que cada estatística nos diz:**

- **Média**: o valor "típico", sensível a valores extremos (*outliers*).
- **Mediana**: o valor central quando os dados são ordenados; menos sensível a extremos que a média.
- **Desvio padrão**: o quanto os valores costumam variar em torno da média. Desvio padrão alto = leituras mais instáveis.
- **Mínimo/Máximo**: os extremos observados — úteis para identificar leituras fora do esperado.

## 2. Estatísticas em matrizes, por linha ou coluna

Assim como vimos com `.mean(axis=...)` na aula anterior, todas essas funções aceitam `axis`.

In [8]:
leituras_por_equipamento = np.array([
    [72.0, 74.5, 76.0, 75.2],   # Motor-01
    [65.2, 66.8, 64.1, 65.9],   # Bomba-03
    [88.0, 90.5, 91.2, 89.7],   # Compressor-02
])

print("Desvio padrão por equipamento (axis=1):", np.round(leituras_por_equipamento.std(axis=1), 2))
print("Máximo por instante (axis=0):          ", leituras_por_equipamento.max(axis=0))


Desvio padrão por equipamento (axis=1): [1.5  0.99 1.19]
Máximo por instante (axis=0):           [88.  90.5 91.2 89.7]


## 3. Valores ausentes: `NaN`

Em dados reais, sensores falham, se desconectam ou enviam leituras corrompidas. Esses "buracos" nos dados são representados por `NaN` (*Not a Number*), disponível como `np.nan`.

In [9]:
leituras_com_falha = np.array([72.0, 76.5, np.nan, 68.3, np.nan, 74.1])

print(leituras_com_falha)
print("dtype:", leituras_com_falha.dtype)   # arrays com NaN são sempre float


[72.  76.5  nan 68.3  nan 74.1]
dtype: float64


**Cuidado:** operações estatísticas comuns "contaminam" o resultado quando há `NaN` no array — qualquer cálculo que inclua um `NaN` também resulta em `NaN`.

In [10]:
leituras_com_falha = np.array([72.0, 76.5, np.nan, 68.3, np.nan, 74.1])

print("Média comum (incorreta na presença de NaN):", leituras_com_falha.mean())


Média comum (incorreta na presença de NaN): nan


## 4. Identificando valores ausentes

A função `np.isnan()` retorna `True`/`False` para cada posição, indicando se o valor é `NaN`. Combinada com indexação booleana, permite localizar e contar as falhas.

In [11]:
leituras_com_falha = np.array([72.0, 76.5, np.nan, 68.3, np.nan, 74.1])

mascara_ausente = np.isnan(leituras_com_falha)
print("Máscara de valores ausentes:", mascara_ausente)
print("Quantidade de leituras ausentes:", mascara_ausente.sum())   # True conta como 1
print("Posições com falha:", np.where(mascara_ausente)[0])


Máscara de valores ausentes: [False False  True False  True False]
Quantidade de leituras ausentes: 2
Posições com falha: [2 4]


**Adicional sobre `np.where`**

np.where é uma função do NumPy usada para localizar posições que satisfazem uma condição ou para escolher valores com base em uma condição.

```py
np.where(condição, valor_se_verdadeiro, valor_se_falso)
```

Então podemos pensar em np.where como um if vetorizado:

In [12]:
import numpy as np

a = np.array([10, 25, 8, 40])

resultado = np.where(a >= 20, "Maior que 20", "Menor que 20")

for item, texto in zip(a, resultado):
    print(f"{item}: {texto}")

10: Menor que 20
25: Maior que 20
8: Menor que 20
40: Maior que 20


## 5. Calculando estatísticas ignorando os `NaN`

O NumPy oferece uma versão de cada função estatística com o prefixo `nan`, que **ignora** os valores ausentes automaticamente.

In [13]:
leituras_com_falha = np.array([72.0, 76.5, np.nan, 68.3, np.nan, 74.1])

print("Média ignorando NaN:   ", round(np.nanmean(leituras_com_falha), 2))
print("Mediana ignorando NaN: ", np.nanmedian(leituras_com_falha))
print("Máximo ignorando NaN:  ", np.nanmax(leituras_com_falha))


Média ignorando NaN:    72.72
Mediana ignorando NaN:  73.05
Máximo ignorando NaN:   76.5


## 6. Um primeiro tratamento: substituir `NaN` por um valor

Uma abordagem simples (vamos aprofundar isso com o Pandas no Módulo 4) é substituir os valores ausentes pela média das leituras válidas, usando a máscara booleana.

In [14]:
leituras_com_falha = np.array([72.0, 76.5, np.nan, 68.3, np.nan, 74.1])

leituras_tratadas = leituras_com_falha.copy()          # trabalhamos em uma cópia, preservando o original
media_valida = np.nanmean(leituras_com_falha)
leituras_tratadas[np.isnan(leituras_tratadas)] = media_valida

print("Original: ", leituras_com_falha)
print("Tratado:  ", np.round(leituras_tratadas, 2))


Original:  [72.  76.5  nan 68.3  nan 74.1]
Tratado:   [72.   76.5  72.72 68.3  72.72 74.1 ]


## 7. Resumo da aula

- NumPy calcula estatísticas descritivas prontas: `sum`, `mean`, `median` (`np.median`), `min`, `max`, `std`, `var`.
- Todas aceitam `axis` para calcular por linha ou por coluna em matrizes.
- `np.nan` representa um valor ausente; qualquer cálculo comum que o inclua resulta em `NaN`.
- `np.isnan()` localiza valores ausentes; as versões `np.nanmean()`, `np.nanmedian()`, `np.nanmax()` etc. ignoram-nos automaticamente.
- Uma forma simples de tratamento é substituir os `NaN` por uma estatística calculada sobre os valores válidos.

### Exercício sugerido

Crie um array `pressoes` com 8 valores, incluindo 2 `np.nan`. Calcule a média ignorando os ausentes e substitua os `NaN` por essa média, imprimindo o array antes e depois do tratamento.
